# Module 5.1: Hierarchical Forecasting

This notebook demonstrates hierarchical forecasting and reconciliation for:

**Category → Subcategory → SKU**

We will:
- aggregate history to multiple levels
- create simple example forecasts
- reconcile forecasts (bottom-up vs top-down)
- validate that forecasts are consistent across levels


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../')

from src.data.loaders import load_sales_data
from src.hierarchy import (
    HierarchySpec,
    aggregate_to_level,
    bottom_up_reconcile,
    top_down_reconcile,
    reconcile_to_category_totals,
)
from src.models.baseline import forecast_by_sku

print('Imports OK')


Imports OK


## Load data

We use raw sales data with hierarchy columns (`category`, `subcategory`, `sku_id`).


In [2]:
raw_candidates = [Path('../data/raw/sample_sales.csv'), Path('data/raw/sample_sales.csv')]
raw_path = next((p for p in raw_candidates if p.exists()), None)
if raw_path is None:
    raise FileNotFoundError('Could not find data/raw/sample_sales.csv')

df = load_sales_data(raw_path)
print('Loaded:', raw_path, df.shape)
df[['date','sku_id','category','subcategory','units_sold']].head()


Loaded: ..\data\raw\sample_sales.csv (365000, 9)


,date,sku_id,category,subcategory,units_sold
0,2023-12-18,SKU001,Electronics,Accessories,19
1,2023-12-18,SKU002,Electronics,Audio,0
2,2023-12-18,SKU003,Electronics,Accessories,25
3,2023-12-18,SKU004,Electronics,Phones,100
4,2023-12-18,SKU005,Electronics,Laptops,71


## Aggregate to hierarchy levels

We create time series at three levels:
- category
- subcategory
- sku


In [3]:
spec = HierarchySpec()

cat_hist = aggregate_to_level(df, spec, level='category')
sub_hist = aggregate_to_level(df, spec, level='subcategory')
sku_hist = aggregate_to_level(df, spec, level='sku')

print('Category history:', cat_hist.shape)
print('Subcategory history:', sub_hist.shape)
print('SKU history:', sku_hist.shape)

cat_hist.head()


Category history: (3650, 3)
Subcategory history: (14600, 4)
SKU history: (365000, 5)


,date,category,units_sold
0,2023-12-18,Books,3854
1,2023-12-18,Clothing,3549
2,2023-12-18,Electronics,3987
3,2023-12-18,Home & Garden,3573
4,2023-12-18,Sports,3653


## Create example forecasts

To demonstrate reconciliation, we need forecasts at different levels.

We’ll do:
- **SKU-level forecasts** using a simple baseline (moving average)
- **Category-level forecasts** using a naive baseline

Then we’ll reconcile and validate consistency.


In [4]:
HORIZON = 14
cutoff = df['date'].max() - pd.Timedelta(days=HORIZON)

# SKU-level forecasts (local baseline)
sku_train = df[df['date'] <= cutoff].copy()
sku_fc = forecast_by_sku(
    sku_train,
    sku_col='sku_id',
    date_col='date',
    target_col='units_sold',
    horizon=HORIZON,
    method='moving_average',
    ma_window=7,
    cutoff_date=cutoff,
)

# Add category/subcategory mapping to SKU forecasts
mapping = df[['sku_id','category','subcategory']].drop_duplicates('sku_id')
sku_fc = sku_fc.merge(mapping, on='sku_id', how='left')

# Category-level forecasts (naive baseline on aggregated category series)
cat_train = cat_hist[cat_hist['date'] <= cutoff].copy()
cat_rows = []
for cat, g in cat_train.groupby('category'):
    y = g.sort_values('date')['units_sold'].to_numpy(dtype=float)
    last = float(y[-1])
    future_dates = pd.date_range(start=cutoff + pd.Timedelta(days=1), periods=HORIZON, freq='D')
    for d, p in zip(future_dates, [last]*HORIZON):
        cat_rows.append({'date': d, 'category': cat, 'y_pred': p})
cat_fc = pd.DataFrame(cat_rows)

print('SKU forecasts:', sku_fc.shape)
print('Category forecasts:', cat_fc.shape)
sku_fc.head()


SKU forecasts: (7000, 6)
Category forecasts: (70, 3)


,sku_id,date,y_pred,method,category,subcategory
0,SKU001,2025-12-03,26.428571,moving_average,Electronics,Accessories
1,SKU001,2025-12-04,26.428571,moving_average,Electronics,Accessories
2,SKU001,2025-12-05,26.428571,moving_average,Electronics,Accessories
3,SKU001,2025-12-06,26.428571,moving_average,Electronics,Accessories
4,SKU001,2025-12-07,26.428571,moving_average,Electronics,Accessories


## Reconciliation

We’ll show three approaches:

1. **Bottom-up**: Aggregate SKU forecasts to subcategory/category.
2. **Top-down**: Allocate category forecasts down to SKUs using historical proportions.
3. **Scale-to-total**: Scale SKU forecasts so category totals match the category forecast.


In [5]:
# 1) Bottom-up
sub_bu, cat_bu = bottom_up_reconcile(sku_fc, spec)

# 2) Top-down (category -> sku)
sku_td = top_down_reconcile(cat_fc, history=df[df['date'] <= cutoff], spec=spec, from_level='category', to_level='sku', window_days=90, as_of=cutoff)

# 3) Scale-to-total reconciliation
sku_scaled = reconcile_to_category_totals(sku_fc[[ 'date','category','subcategory','sku_id','y_pred' ]], cat_fc, spec)

print('Bottom-up category rows:', cat_bu.shape)
print('Top-down SKU rows:', sku_td.shape)
print('Scaled SKU rows:', sku_scaled.shape)

cat_bu.head()


Bottom-up category rows: (70, 3)
Top-down SKU rows: (7000, 5)
Scaled SKU rows: (7000, 5)


,date,category,y_pred_category
0,2025-12-03,Books,4720.857143
1,2025-12-03,Clothing,4269.142857
2,2025-12-03,Electronics,4753.285714
3,2025-12-03,Home & Garden,4891.714286
4,2025-12-03,Sports,4850.571429


## Consistency checks

We verify whether category totals match:

- Bottom-up category totals should equal the sum of SKU forecasts.
- Top-down allocations should sum back to the category forecast.
- Scaled SKU forecasts should match the category forecast by construction.


In [6]:
def max_abs_diff(a: pd.Series, b: pd.Series) -> float:
    return float((a - b).abs().max())

# Bottom-up: compare category totals from cat_bu vs direct sum from sku_fc
sku_sum = sku_fc.groupby(['date','category'], as_index=False)['y_pred'].sum().rename(columns={'y_pred':'sum_sku'})
cat_bu2 = cat_bu.merge(sku_sum, on=['date','category'], how='left')
print('Bottom-up max abs diff (should be ~0):', max_abs_diff(cat_bu2['y_pred_category'], cat_bu2['sum_sku']))

# Top-down: sum allocated SKUs and compare to category forecast
td_sum = sku_td.groupby(['date','category'], as_index=False)['y_pred'].sum().rename(columns={'y_pred':'sum_alloc'})
cat_td2 = cat_fc.merge(td_sum, on=['date','category'], how='left')
print('Top-down max abs diff (should be ~0):', max_abs_diff(cat_td2['y_pred'], cat_td2['sum_alloc']))

# Scaled: sum scaled SKUs and compare to category forecast
scaled_sum = sku_scaled.groupby(['date','category'], as_index=False)['y_pred'].sum().rename(columns={'y_pred':'sum_scaled'})
cat_sc2 = cat_fc.merge(scaled_sum, on=['date','category'], how='left')
print('Scaled max abs diff (should be ~0):', max_abs_diff(cat_sc2['y_pred'], cat_sc2['sum_scaled']))


Bottom-up max abs diff (should be ~0): 0.0
Top-down max abs diff (should be ~0): 1.3642420526593924e-11
Scaled max abs diff (should be ~0): 9.849827620200813e-10


## Quick comparison: bottom-up vs top-down allocations

For one category, we’ll compare how SKU allocations differ between:
- bottom-up (SKU model-driven)
- top-down (proportion-driven)

This helps build intuition about when each method is appropriate.


In [7]:
cat_example = df['category'].value_counts().index[0]

bu_cat_total = sku_fc[sku_fc['category'] == cat_example].groupby('sku_id')['y_pred'].sum().sort_values(ascending=False)
td_cat_total = sku_td[sku_td['category'] == cat_example].groupby('sku_id')['y_pred'].sum().sort_values(ascending=False)

comparison = pd.DataFrame({
    'bottom_up_total': bu_cat_total,
    'top_down_total': td_cat_total,
}).fillna(0)
comparison['diff'] = comparison['bottom_up_total'] - comparison['top_down_total']
comparison = comparison.sort_values('bottom_up_total', ascending=False)

print('Category example:', cat_example)
comparison.head(10)


Category example: Electronics


,bottom_up_total,top_down_total,diff
sku_id,,,
SKU081,1604.0,941.390527,662.609473
SKU045,1402.0,985.573754,416.426246
SKU088,1304.0,949.854747,354.145253
SKU083,1278.0,967.798893,310.201107
SKU017,1278.0,932.587738,345.412262
SKU100,1224.0,992.514415,231.485585
SKU056,1198.0,980.495222,217.504778
SKU091,1166.0,920.399262,245.600738
SKU065,1162.0,989.298011,172.701989
